>En este notebook esta mejor correrse con GPU. En Colab, anda a Entorno de >ejecucion > Cambiar tipo de entorno de ejecucion > GPU antes de empezar.

# Ejercicio: Armando un Filtro de Spam con BERT

En este ejercicio vas a aplicar exactamente las mismas fases, con la misma receta (BERT congelado, `Trainer`, etc), pero sobre un problema distinto: **detectar mensajes de texto (SMS) que son spam**.

Es el mismo tipo de tarea que resuelven de verdad las aplicaciones de mensajes cuando te avisan "esto podria ser spam": clasificacion binaria de texto, entrenada sobre ejemplos reales etiquetados.

### Un detalle importante antes de empezar: el dataset esta desbalanceado

De los 5,574 mensajes del dataset, la gran mayoria son mensajes legitimos (**ham**) y solo una fraccion chica son spam. Ya nos encontramos con esta situacion antes: si no le prestamos atencion, un modelo puede aprender el atajo de "predecir siempre ham" y aun asi mostrar un accuracy que parece alto, sin haber aprendido a detectar spam en absoluto. Vas a tener que manejar este desbalance vos mismo en este ejercicio, con las mismas herramientas que ya usamos.

### El plan

1. Fase 1: preparar el entorno, cargar BERT y su tokenizador.
2. Fase 2: cargar el dataset de SMS, revisar el balance de clases, y prepararlo para el modelo.
3. Fase 3: congelar las capas base de BERT.
4. Fase 4: entrenar, evaluar (con atencion especial al desbalance) y probar el modelo con un mensaje nuevo.

## Fase 1: Configuracion del Entorno y el Modelo

In [23]:
# Quitamos torchvision: no la necesitamos (solo trabajamos con texto), y en algunos entornos
# de Colab su version rompe la deteccion automatica que hace la libreria datasets
!pip uninstall -y -q torchvision
!pip install -q transformers datasets accelerate scikit-learn evaluate

In [24]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from datasets import load_dataset, Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
)

print("GPU disponible:", torch.cuda.is_available())

GPU disponible: True


Cargamos el mismo modelo base que usamos con los tweets: `bert-base-multilingual-cased`, junto con su tokenizador.

In [25]:
MODEL_NAME = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Modelo seleccionado:", MODEL_NAME)
print("Tokenizador cargado correctamente.")

Modelo seleccionado: bert-base-multilingual-cased
Tokenizador cargado correctamente.


## Fase 2: Preparacion del Dataset

Esta vez, en vez de bajar un CSV con pandas, cargamos el dataset directamente desde el Hugging Face Hub con `load_dataset`. Es la forma mas comun de conseguir datasets ya preparados para trabajar con la libreria `datasets`, sin tener que limpiar ni convertir nada a mano.

In [26]:
dataset_sms = load_dataset("ucirvine/sms_spam")

df = dataset_sms["train"].to_pandas()

# Estandarizamos el nombre de la columna de texto para mantener consistencia
# con el resto del notebook (label ya viene como 0=ham, 1=spam).
df = df.rename(columns={"sms": "texto"})
df["texto"] = df["texto"].astype(str).str.strip()

print("Columnas:", df.columns.tolist())
print("Registros totales:", len(df))
df.head()

Columnas: ['texto', 'label']
Registros totales: 5574


,texto,label
0,"Go until jurong point, crazy.. Available only ...",0
1,Ok lar... Joking wif u oni...,0
2,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,U dun say so early hor... U c already then say...,0
4,"Nah I don't think he goes to usf, he lives aro...",0


Revisemos el balance de clases antes de hacer cualquier otra cosa. `label` vale 0 para mensajes legitimos (ham) y 1 para spam.

In [27]:
print("Distribución de clases (0 = ham, 1 = spam):")
display(df["label"].value_counts())

print()
print("Porcentaje por clase:")
display((df["label"].value_counts(normalize=True) * 100).round(2))

Distribución de clases (0 = ham, 1 = spam):


,count
label,
0,4827
1,747



Porcentaje por clase:


,proportion
label,
0,86.6
1,13.4


Como esperabamos, el dataset esta lejos de estar balanceado. Si entrenaramos directamente sobre esta distribucion, corremos el riesgo de que el modelo aprenda a predecir "ham" siempre y aun asi consiga un accuracy alto, sin haber aprendido nada util sobre como se ve un mensaje de spam.

Armemos una muestra balanceada, con la misma cantidad de mensajes de cada clase (por ejemplo 700).

In [28]:
n_por_clase = 700

df_ham = df[df["label"] == 0].sample(n=n_por_clase, random_state=42)
df_spam = df[df["label"] == 1].sample(n=n_por_clase, random_state=42)

df_balanceado = (
    pd.concat([df_ham, df_spam])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("Registros balanceados:", len(df_balanceado))
display(df_balanceado["label"].value_counts())

Registros balanceados: 1400


,count
label,
0,700
1,700


Separamos en train y test, manteniendo la proporcion de clases en ambos conjuntos con `stratify`.

In [29]:
train_df, test_df = train_test_split(
    df_balanceado,
    test_size=0.20,
    stratify=df_balanceado["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Entrenamiento:", len(train_df))
print("Test:", len(test_df))
print()
print("Distribución en entrenamiento:")
display(train_df["label"].value_counts())
print()
print("Distribución en test:")
display(test_df["label"].value_counts())

Entrenamiento: 1120
Test: 280

Distribución en entrenamiento:


,count
label,
0,560
1,560



Distribución en test:


,count
label,
0,140
1,140


Convertimos a `Dataset` de Hugging Face y tokenizamos en batch. Por eso definimos un funcion "tokenizar". Los mensajes de SMS son cortos, asi que un `max_length` chico alcanza (por ejemplo max_length=64).

In [30]:
# Función que tokeniza
def tokenizar(batch):
    return tokenizer(
        batch["texto"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

# Convertimos los DataFrames en datasets compatibles con Hugging Face
train_dataset = Dataset.from_pandas(train_df[["texto", "label"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["texto", "label"]], preserve_index=False)

# Tokenizamos en batch y quitamos la columna de texto original.
# NO usamos set_format("torch"): el Trainer convierte los batches a tensores automáticamente.
train_dataset = train_dataset.map(tokenizar, batched=True, remove_columns=["texto"])
test_dataset = test_dataset.map(tokenizar, batched=True, remove_columns=["texto"])

print("Dataset de entrenamiento:")
print(train_dataset)
print("\nDataset de test:")
print(test_dataset)

ejemplo = train_dataset[0]
print("\nClaves del primer ejemplo tokenizado:", ejemplo.keys())
print("Label:", ejemplo["label"])
print("Primeros 12 input_ids:", ejemplo["input_ids"][:12])

Map:   0%|          | 0/1120 [00:00<?, ? examples/s]

Map:   0%|          | 0/280 [00:00<?, ? examples/s]

Dataset de entrenamiento:
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1120
})

Dataset de test:
Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 280
})

Claves del primer ejemplo tokenizado: dict_keys(['label', 'input_ids', 'token_type_ids', 'attention_mask'])
Label: 0
Primeros 12 input_ids: [101, 50133, 13028, 15329, 11178, 20592, 136, 10747, 10924, 177, 34420, 10123]


## Fase 3: Implementacion de Transfer Learning

Cargamos BERT con una cabeza de clasificacion nueva para dos categorias, y aplicamos el paso critico: congelar las capas base con `requires_grad = False`, para entrenar unicamente la cabeza de clasificacion.

In [31]:
label2id = {"ham": 0, "spam": 1}
id2label = {0: "ham", 1: "spam"}

num_labels = len(label2id)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Congelar las capas base de BERT.
# La cabeza de clasificación permanece entrenable.
for param in model.base_model.parameters():
    param.requires_grad = False

print("Modelo cargado y capas base congeladas.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado y capas base congeladas.


Verifiquemos el efecto del congelado, igual que hicimos antes.  
Muestre al menos 5 decimales para ver correctamente el porcentaje de parametro entrenable.

In [32]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parámetros totales: {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Porcentaje entrenable: {100 * trainable_params / total_params:.5f}%")

print("\nCapas que permanecen entrenables:")
for nombre, parametro in model.named_parameters():
    if parametro.requires_grad:
        print("-", nombre)

Parámetros totales: 177,854,978
Parámetros entrenables: 1,538
Porcentaje entrenable: 0.00086%

Capas que permanecen entrenables:
- classifier.weight
- classifier.bias


## Fase 4: Entrenamiento y Evaluacion

### Midiendo el desempeño, con atencion al desbalance

Ya balanceamos el dataset antes de entrenar, asi que el accuracy deberia ser un numero confiable esta vez. Aun asi, para un filtro de spam vale la pena mirar mas de cerca: **precision** y **recall** de la clase spam en particular.

- **Recall de spam**: de todos los mensajes que realmente eran spam, ¿que porcentaje detecto el modelo? Un recall bajo significa que se le esta escapando spam.
- **Precision de spam**: de todos los mensajes que el modelo marco como spam, ¿que porcentaje realmente lo era? Una precision baja significa que el modelo esta marcando mensajes legitimos como spam por error.

En un filtro de spam real, estos dos errores no son igual de graves: dejar pasar algo de spam es molesto, pero marcar un mensaje legitimo como spam puede hacer que la persona se pierda algo importante. Por eso no alcanza con mirar el accuracy solo.

In [33]:
# Funcion para calcular métrica
metrica_accuracy = evaluate.load("accuracy")

def calcular_metricas(eval_pred):
    logits, labels = eval_pred
    predicciones = np.argmax(logits, axis=-1)

    acc = metrica_accuracy.compute(predictions=predicciones, references=labels)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predicciones, average=None, labels=[0, 1], zero_division=0
    )

    return {
        "accuracy": acc["accuracy"],
        "precision_spam": precision[1],
        "recall_spam": recall[1],
        "f1_spam": f1[1],
    }

Configuramos `TrainingArguments`.

In [34]:
argumentos_entrenamiento = TrainingArguments(
    output_dir="./resultados_bert_spam",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=1e-3,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42
)

Armamos el `Trainer` y entrenamos.

In [35]:
trainer = Trainer(
    model=model,
    args=argumentos_entrenamiento,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=calcular_metricas,
    processing_class=tokenizer
)

resultado_entrenamiento = trainer.train()
resultado_entrenamiento

Epoch,Training Loss,Validation Loss,Accuracy,Precision Spam,Recall Spam,F1 Spam
1,0.583647,0.460279,0.885714,0.821429,0.985714,0.896104
2,0.423865,0.373108,0.935714,0.969231,0.900000,0.933333
3,0.378770,0.348537,0.928571,0.910959,0.950000,0.930070


TrainOutput(global_step=210, training_loss=0.4620941434587751, metrics={'train_runtime': 19.0758, 'train_samples_per_second': 176.139, 'train_steps_per_second': 11.009, 'total_flos': 110506643251200.0, 'train_loss': 0.4620941434587751, 'epoch': 3.0})

Evaluamos el modelo entrenado sobre el conjunto de test.

In [36]:
resultados = trainer.evaluate()

print("Resultados de evaluación:")
for clave, valor in resultados.items():
    if isinstance(valor, float):
        print(f"{clave}: {valor:.4f}")
    else:
        print(f"{clave}: {valor}")

Training Loss,Validation Loss,Epoch,Accuracy,Precision Spam,Recall Spam,F1 Spam
0.378770,0.348537,3,0.928571,0.910959,0.950000,0.930070


Resultados de evaluación:
eval_loss: 0.3485
eval_accuracy: 0.9286
eval_precision_spam: 0.9110
eval_recall_spam: 0.9500
eval_f1_spam: 0.9301


Una posible mejora del modelo es experimentar con el número de épocas y el learning rate. Si BERT está congelado, podemos utilizar un learning rate mayor para facilitar el aprendizaje de la capa de regresión.  
<br>
Es importante comparar las métricas de validación para comprobar si estas modificaciones realmente mejoran el modelo y evitar overfitting.

### Mirando la matriz de confusion

Los cuatro numeros de una matriz de confusion cuentan la historia completa: cuantos mensajes de cada clase el modelo clasifico bien, y en que direccion se equivoco cuando fallo.

In [37]:
predicciones_test = trainer.predict(test_dataset)
y_pred = np.argmax(predicciones_test.predictions, axis=-1)
y_true = predicciones_test.label_ids

matriz = confusion_matrix(y_true, y_pred, labels=[0, 1])

print("Matriz de confusión (filas=real, columnas=predicho) [ham, spam]:")
print(matriz)

tn, fp, fn, tp = matriz.ravel()
print(f"\nHam real / Ham predicho:   {tn}")
print(f"Ham real / Spam predicho:  {fp}  (falsos positivos)")
print(f"Spam real / Ham predicho:  {fn}  (falsos negativos -- spam que se escapo)")
print(f"Spam real / Spam predicho: {tp}")

Matriz de confusión (filas=real, columnas=predicho) [ham, spam]:
[[127  13]
 [  7 133]]

Ham real / Ham predicho:   127
Ham real / Spam predicho:  13  (falsos positivos)
Spam real / Ham predicho:  7  (falsos negativos -- spam que se escapo)
Spam real / Spam predicho: 133


Fijate en particular la celda "Spam real / Ham predicho": esos son los mensajes de spam que se le escaparon al modelo. Y la celda "Ham real / Spam predicho": esos son mensajes legitimos marcados como spam por error. Compara esos dos numeros con el `recall_spam` y `precision_spam` que calculamos antes -- deberian contar la misma historia.

## Inferencia con un mensaje nuevo

Probemos el modelo con un mensaje que armamos nosotros, que nunca vio durante el entrenamiento.  


In [38]:
# Primero, escribe una función para recibir un texto y predecir si es spam o no, devolviendo también la probabilidad de la predicción.
def predecir_spam(texto, modelo=model, tokenizador=tokenizer):
    modelo.eval()

    encoding = tokenizador(
        texto,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    encoding = {k: v.to(modelo.device) for k, v in encoding.items()}

    with torch.no_grad():
        outputs = modelo(**encoding)

    probabilidades = torch.softmax(outputs.logits, dim=-1)
    clase_id = torch.argmax(probabilidades, dim=-1).item()
    score = probabilidades[0][clase_id].item()
    categoria = modelo.config.id2label[clase_id]

    return categoria, score

Proba con tus propios mensajes:

In [39]:
mensajes_prueba = [
    "Congratulations! You've WON a $1000 Walmart gift card. Click here to claim now!!!",
    "Hey, are we still meeting for lunch tomorrow at 1pm?",
    "URGENT: Your account has been suspended. Verify your identity immediately at this link.",
    "Nos vemos manana en la oficina para revisar el informe.",
]

for mensaje in mensajes_prueba:
    categoria, score = predecir_spam(mensaje)
    print("Mensaje:", mensaje)
    print("Categoria ganadora:", categoria)
    print(f"Confianza: {score:.2%}")
    print("-" * 70)

Mensaje: Congratulations! You've WON a $1000 Walmart gift card. Click here to claim now!!!
Categoria ganadora: ham
Confianza: 54.64%
----------------------------------------------------------------------
Mensaje: Hey, are we still meeting for lunch tomorrow at 1pm?
Categoria ganadora: ham
Confianza: 73.83%
----------------------------------------------------------------------
Mensaje: URGENT: Your account has been suspended. Verify your identity immediately at this link.
Categoria ganadora: spam
Confianza: 61.84%
----------------------------------------------------------------------
Mensaje: Nos vemos manana en la oficina para revisar el informe.
Categoria ganadora: ham
Confianza: 52.33%
----------------------------------------------------------------------


Proba con tus propios mensajes: alguno claramente spam (premios, urgencia, links sospechosos), alguno claramente legitimo, y alguno ambiguo. Fijate tambien que pasa si escribis el mensaje en español -- el modelo es multilingue, pero fue entrenado unicamente con ejemplos en ingles, asi que es un buen caso para pensar los limites de lo que aprendio.

## Resumen

- Trabajamos sobre un dataset genuinamente desbalanceado (13% spam, 87% ham), y lo corregimos armando una muestra balanceada antes de entrenar, con la misma tecnica que ya conociamos.
- Cargamos el dataset directamente desde el Hugging Face Hub con `load_dataset`, en vez de partir de un CSV.
- Aplicamos las cuatro fases del temario sobre un problema nuevo: setup del modelo, preparacion del dataset, congelado de capas, y entrenamiento con `Trainer`.
- Fuimos un paso mas alla del accuracy: miramos precision y recall especificamente para la clase spam, y una matriz de confusion, porque en un filtro de spam real no todos los errores pesan lo mismo.
- Probamos el modelo final con mensajes nuevos, incluyendo un caso fuera del idioma de entrenamiento.